# Part 2: Sparse Attention Vectors (SAVs) on CC-OCR doc/table parsing (Colab, T4)

**Read `PROJECT_CONTEXT.md` first.** Everyone runs this same notebook; only `MODEL_KEY` changes.

| MODEL_KEY | Model | Owner |
|---|---|---|
| `qwen25vl_7b` | Qwen/Qwen2.5-VL-7B-Instruct (4-bit) | Ujjwal |
| `phi4_mm` | microsoft/Phi-4-multimodal-instruct (fp16) | Ujjwal |
| `llama32v_11b` | meta-llama/Llama-3.2-11B-Vision-Instruct (4-bit, gated) | Anvesh |
| `gemma3_12b` | google/gemma-3-12b-it (4-bit, gated) | Pratham |

**Runtime → Change runtime type → T4 GPU.** Every phase is resumable: if Colab disconnects, run all cells again and it continues where it stopped (outputs live on your Google Drive).

In [ ]:
# ===== 1. CONFIG: the only cell you should edit =====
MODEL_KEY = "qwen25vl_7b"          # qwen25vl_7b | phi4_mm | llama32v_11b | gemma3_12b
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/BTP"   # repo is cloned here so results persist on Drive
RUN_PART_2A = True                 # SAV extraction + zero-shot (~1.5-3 h per model)
RUN_PART_2B = True                 # full parsing under 3 configs (slow; resumable)
N_PARSE_PER_SPLIT = 25             # images per split for Part 2B (25 x 4 splits x 3 configs = 300 generations)
DTYPE_OVERRIDE = None              # e.g. "float32" if you get NaN / FloatingPointError
REPO_URL = "https://github.com/UjjWaL-0911/Evaluation-Of-MultiModal-LLMs-for-Layout-Aware-Document-Parsing.git"

In [ ]:
# ===== 2. Mount Drive and get the code =====
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
REPO_DIR = os.path.join(DRIVE_PROJECT_DIR, "Evaluation-Of-MultiModal-LLMs-for-Layout-Aware-Document-Parsing")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
# ===== 3. Install dependencies (Phi-4 needs old pinned versions) =====
if MODEL_KEY == "phi4_mm":
    %pip install -q "transformers==4.48.2" "peft==0.13.2" "accelerate==1.3.0" soundfile scipy backoff
else:
    %pip install -q -U "transformers>=4.50" accelerate bitsandbytes
%pip install -q rapidfuzz apted distance lxml tabulate nltk
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4!")

**Gated models (Llama-3.2-Vision, Gemma-3):** first accept the license on the model's Hugging Face page with your
account, create a *read* token at huggingface.co/settings/tokens, and add it in Colab's 🔑 **Secrets** panel as
`HF_TOKEN` (enable notebook access). Qwen and Phi-4 do not need this but it still speeds up downloads.

In [ ]:
# ===== 4. Hugging Face login =====
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    print("Logged in to Hugging Face.")
except Exception as e:
    print("No HF_TOKEN secret found (required for llama32v_11b and gemma3_12b):", e)

In [ ]:
# ===== 5. Load model + attention-head hooks, sanity check on one image =====
import time
from savs.models import load_model, MODELS
from savs.extract import HeadExtractor
from savs.pipeline import run_extraction, run_parsing, write_run_info, progress
from savs.tasks import TASKS, PARSE_CONFIGS, EXTRACT_MAX_EDGE, task_prompt
from savs.preprocess import preprocess
from dataset import load_split, decode_image

t0 = time.time()
vlm = load_model(MODEL_KEY, dtype=DTYPE_OVERRIDE)
ext = HeadExtractor(vlm)
print(f"Loaded {MODELS[MODEL_KEY]['hf_id']} in {time.time()-t0:.0f}s | {ext.describe()}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(write_run_info(vlm, ext))

row = load_split("Photoed_Docs", 1).iloc[0]
img = preprocess(decode_image(row["image"]), max_edge=EXTRACT_MAX_EDGE)
ext.reset(); t0 = time.time()
ans = vlm.run(img, task_prompt("acquisition", row), max_new_tokens=8)
acts = ext.collect()
print(f"Zero-shot answer: {ans!r} (truth: photographed) | activations {tuple(acts.shape)} | {time.time()-t0:.1f}s")

In [ ]:
# ===== 6. PART 2A: SAV activations + zero-shot answers for all classification tasks (+ 'parse') =====
if RUN_PART_2A:
    for task in TASKS:
        run_extraction(vlm, ext, task)
progress(MODEL_KEY, n_parse_per_split=N_PARSE_PER_SPLIT)

In [ ]:
# ===== 7. Quick look at YOUR model's Part 2A results (CPU, ~1 min) =====
from savs.analysis import analyze, to_markdown
from IPython.display import Markdown, display
display(Markdown(to_markdown(analyze([MODEL_KEY]))))

In [ ]:
# ===== 8. PART 2B: full parsing under each pre-processing config (slow, resumable) =====
# Rough cost on a T4: 1-3 min per image. Watch the first few timings and lower N_PARSE_PER_SPLIT if needed
# (but tell the team: everyone should use the same N).
if RUN_PART_2B:
    for cfg in PARSE_CONFIGS:
        run_parsing(vlm, cfg, n_per_split=N_PARSE_PER_SPLIT)
progress(MODEL_KEY, n_parse_per_split=N_PARSE_PER_SPLIT)
display(Markdown(to_markdown(analyze([MODEL_KEY]))))

In [ ]:
# ===== 9. Hand-off: zip your results so Ujjwal can combine all models =====
import shutil
zip_path = shutil.make_archive(f"/content/drive/MyDrive/SAVs_results_{MODEL_KEY}", "zip",
                               root_dir="Evaluation_Results/SAVs", base_dir=MODEL_KEY)
print("Upload / share this file with Ujjwal:", zip_path)